In [1]:
import json
import os

data = []

with open(r'Datasets/Experiment_2_input_sample.json', 'r') as file:
    for line in file:
        data.append(json.loads(line))

print(data)


[{'No.': 46, 'Time': 3.486046, 'Source': '0x1de6', 'Destination': 'Broadcast', 'Protocol': 'ZigBee', 'Length': 80, 'Info': 'Link Status', 'Info_clean': 'Link Status'}, {'No.': 177, 'Time': 11.472175, 'Source': '0xd7a7', 'Destination': '0x1de6', 'Protocol': 'ZigBee HA', 'Length': 56, 'Info': 'ZCL: Read Attributes, Seq: 216', 'Info_clean': 'ZCL: Read Attributes,'}, {'No.': 179, 'Time': 11.50384, 'Source': '0xd7a7', 'Destination': '0x1de6', 'Protocol': 'ZigBee HA', 'Length': 52, 'Info': 'ZCL: Read Attributes, Seq: 217', 'Info_clean': 'ZCL: Read Attributes,'}, {'No.': 181, 'Time': 11.527324, 'Source': '0x1de6', 'Destination': '0xd7a7', 'Protocol': 'ZigBee HA', 'Length': 69, 'Info': 'ZCL: Read Attributes Response, Seq: 216', 'Info_clean': 'ZCL: Read Attributes Response,'}, {'No.': 185, 'Time': 11.559638, 'Source': '0x1de6', 'Destination': '0xd7a7', 'Protocol': 'ZigBee HA', 'Length': 61, 'Info': 'ZCL: Read Attributes Response, Seq: 217', 'Info_clean': 'ZCL: Read Attributes Response,'}, {'No.

In [2]:
""" Change Broadcast with the address """
for item in data:
    if item['Destination'] == 'Broadcast':
        item['Destination'] = '0xfffc'

In [3]:
""" Create sample packets from json file """
Sample_Packets = []

for item in data:
    sample_packets = {
        'time': str(item['Time']),
        'src': item['Source'],
        'dst': item['Destination'],
        'protocol': item['Protocol'],
        'length': str(item['Length']),
        'info': item['Info']
    }
    Sample_Packets.append(sample_packets)
    #print(sample_prompt)
print(Sample_Packets)

[{'time': '3.486046', 'src': '0x1de6', 'dst': '0xfffc', 'protocol': 'ZigBee', 'length': '80', 'info': 'Link Status'}, {'time': '11.472175', 'src': '0xd7a7', 'dst': '0x1de6', 'protocol': 'ZigBee HA', 'length': '56', 'info': 'ZCL: Read Attributes, Seq: 216'}, {'time': '11.50384', 'src': '0xd7a7', 'dst': '0x1de6', 'protocol': 'ZigBee HA', 'length': '52', 'info': 'ZCL: Read Attributes, Seq: 217'}, {'time': '11.527324', 'src': '0x1de6', 'dst': '0xd7a7', 'protocol': 'ZigBee HA', 'length': '69', 'info': 'ZCL: Read Attributes Response, Seq: 216'}, {'time': '11.559638', 'src': '0x1de6', 'dst': '0xd7a7', 'protocol': 'ZigBee HA', 'length': '61', 'info': 'ZCL: Read Attributes Response, Seq: 217'}, {'time': '17.728144', 'src': '0x1de6', 'dst': '0xfffc', 'protocol': 'ZigBee', 'length': '80', 'info': 'Link Status'}, {'time': '24.215512', 'src': '0xd7a7', 'dst': '0x1de6', 'protocol': 'ZigBee HA', 'length': '56', 'info': 'ZCL: Read Attributes, Seq: 239'}, {'time': '24.243231', 'src': '0xd7a7', 'dst': '

In [4]:
Sources = ["0x1de6", "0xd7a7"]
Destinations = ["0x1de6", "0xd7a7", "0xfffc"]
Protocols = ["ZigBee", "ZigBee HA"]

In [5]:
import json
import re

def extract_json_from_markdown(text):
    """
    Extracts the JSON array from a ```json ... ``` code block.
    """
    match = re.search(r"```json\s*(.*?)\s*```", text, flags=re.S)
    if not match:
        raise ValueError("Could not find JSON code block in model output.")

    json_text = match.group(1).strip()  # extract inner JSON
    return json.loads(json_text)        # parse into Python list

# GPT 4.1

In [6]:
import openai
import os
import time
import pandas as pd
import numpy as np

N = list(range(1,11)) # number of trial
computation_time = []
input_tokens = []
output_tokens = []
total_tokens = []


client = openai.OpenAI(api_key=os.environ["OPENAI_KEY"])

prompt = f"""
Examine the following sample ZigBee network traffic data and generate new traffic entries in a similar format.

Sample data: {Sample_Packets}

Rules:
- The format must match the sample exactly, including fields: 'time', 'src', 'dst', 'protocol', 'length', and 'info'.
- The field 'time' is given in seconds format.
- Use src addresses from this list: {Sources}
- Use dst addresses from this list: {Destinations} 
- Use one of the following protocols: {Protocols} 
- Return the generated json in markdown marker ```json ```
- The range of the number of packets generated should be between 90% and 110% of the number of Sample data.

Return output as JSON in the same format, generating new packets of 600 seconds starting from time = 0.
"""

for n in N:
    # Measure start time
    start_time = time.time()
    response = client.chat.completions.create(
        model="gpt-4.1",
        messages=[
            {"role": "system", "content": "You are a realistic ZigBee network traffic generator. You will generate realistic synthetic ZigBee network traffic packets."},
            {"role": "user", "content": prompt}
        ],
        max_tokens= 32768
    )
    
    # Measure end time
    end_time = time.time()

    # Compute elapsed time
    elapsed_time = end_time - start_time
    print(response.choices[0].message.content)

    print("Input tokens used:", response.usage.prompt_tokens)
    print("Output tokens generated:", response.usage.completion_tokens)
    print("Total tokens used:", response.usage.total_tokens)
    print(f"Computation time: {elapsed_time:.2f} seconds")
    
    computation_time.append(elapsed_time)
    input_tokens.append(response.usage.prompt_tokens)
    output_tokens.append(response.usage.completion_tokens)
    total_tokens.append(response.usage.total_tokens)
    
    generated_message = response.choices[0].message.content
    packets = extract_json_from_markdown(generated_message)
    num_samples = len(packets)


    print("Number of generated packages:", num_samples)
    file_path =  fr"Generated_Traffic/New_Experiments/GPT41/GPT41_Exp2_Trial_{n}_10_minute_generated_message.json"

    with open(file_path, "w", encoding="utf-8") as f:
        json.dump(packets, f, indent=2)
        
    #time.sleep(60)
    
df = pd.DataFrame({
    "Trial": N,   # 1..10
    "Computation_Time": computation_time,
    "Input_Tokens": input_tokens,
    "Output_Tokens": output_tokens,
    "Total_Tokens": total_tokens
})

print(df)
results_array = df.to_numpy()
df.to_csv("Generated_Traffic/New_Experiments/GPT41_Exp2_10minute_results.csv", index=False)

```json
[
  {"time": "2.981365", "src": "0x1de6", "dst": "0xfffc", "protocol": "ZigBee", "length": "80", "info": "Link Status"},
  {"time": "7.837242", "src": "0xd7a7", "dst": "0x1de6", "protocol": "ZigBee HA", "length": "56", "info": "ZCL: Read Attributes, Seq: 14"},
  {"time": "7.864954", "src": "0xd7a7", "dst": "0x1de6", "protocol": "ZigBee HA", "length": "52", "info": "ZCL: Read Attributes, Seq: 15"},
  {"time": "7.886011", "src": "0x1de6", "dst": "0xd7a7", "protocol": "ZigBee HA", "length": "69", "info": "ZCL: Read Attributes Response, Seq: 14"},
  {"time": "7.911609", "src": "0x1de6", "dst": "0xd7a7", "protocol": "ZigBee HA", "length": "61", "info": "ZCL: Read Attributes Response, Seq: 15"},
  {"time": "17.551463", "src": "0x1de6", "dst": "0xfffc", "protocol": "ZigBee", "length": "80", "info": "Link Status"},
  {"time": "21.024821", "src": "0xd7a7", "dst": "0x1de6", "protocol": "ZigBee HA", "length": "56", "info": "ZCL: Read Attributes, Seq: 38"},
  {"time": "21.054289", "src": "